# BirdCLEF+ 2026 — CPU pipeline

End-to-end audio classification pipeline (CPU-only):
1. Load metadata (`train.csv`, `taxonomy.csv`).
2. Precompute & cache 5 s log-mel spectrograms to disk (big CPU speedup).
3. Stratified train / val / test split, weighted sampler for class balance.
4. Pretrained **MobileNetV3-Small** backbone + SpecAugment + mixup.
5. Train with cosine LR; evaluate using the official competition metric.
6. Run inference on `test_soundscapes/` → submission.

Set `DATA_DIR` in the Config cell once the dataset finishes downloading.

## 1. Imports & config

In [ ]:
import os, math, random, json, warnings, hashlib
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchaudio
import librosa

warnings.filterwarnings('ignore')
torch.set_num_threads(max(1, os.cpu_count() // 2))

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

# ====== EDIT THIS once your data has finished downloading ======
DATA_DIR = Path(r'data\birdclef-2026')
# ===============================================================

TRAIN_AUDIO_DIR     = DATA_DIR / 'train_audio'
TRAIN_SOUNDSCAPES   = DATA_DIR / 'train_soundscapes'
TEST_SOUNDSCAPES    = DATA_DIR / 'test_soundscapes'
TRAIN_CSV           = DATA_DIR / 'train.csv'
TRAIN_SS_LABELS_CSV = DATA_DIR / 'train_soundscapes_labels.csv'
TAXONOMY_CSV        = DATA_DIR / 'taxonomy.csv'
SAMPLE_SUB_CSV      = DATA_DIR / 'sample_submission.csv'

CACHE_DIR = Path('mel_cache')          # cached log-mel arrays, huge CPU speedup
CACHE_DIR.mkdir(exist_ok=True)

# Audio / spectrogram
SR              = 32_000
CHUNK_SECONDS   = 5
CHUNK_SAMPLES   = SR * CHUNK_SECONDS
N_FFT           = 1024
HOP_LENGTH      = 512
N_MELS          = 128
FMIN, FMAX      = 50, 14_000

# Training
BATCH_SIZE        = 32
EPOCHS            = 4             # reduced from 6 to ensure <90 min with CPU + EfficientNet-B0
LR                = 3e-4
WEIGHT_DECAY      = 1e-4
MAX_FILES_PER_SP  = 25            # bumped; inputs are cached so I/O is cheap
VAL_FRAC          = 0.15
TEST_FRAC         = 0.15
MIXUP_ALPHA       = 0.4           # 0.0 disables mixup
USE_SPEC_AUGMENT  = True

DEVICE = torch.device('cpu')
print('Torch:', torch.__version__, '| device:', DEVICE)

Torch: 2.12.0+cpu | device: cpu


## 2. Load metadata

In [4]:
taxonomy = pd.read_csv(TAXONOMY_CSV)
print('taxonomy rows:', len(taxonomy))
display(taxonomy.head())

SPECIES = taxonomy['primary_label'].tolist()
NUM_CLASSES = len(SPECIES)
label2idx = {s: i for i, s in enumerate(SPECIES)}
print('num classes:', NUM_CLASSES)

train_df = pd.read_csv(TRAIN_CSV)
print('train.csv rows:', len(train_df))
display(train_df.head(3))

# Class distribution
cls_counts = train_df['primary_label'].value_counts()
print('species with train_audio samples:', cls_counts.shape[0])
print('median samples / species:', int(cls_counts.median()))

taxonomy rows: 234


,primary_label,inat_taxon_id,scientific_name,common_name,class_name
0,1161364,1161364,Guyalna cuta,Guyalna cuta,Insecta
1,116570,116570,Caiman yacare,Southern Spectacled Caiman,Reptilia
2,1176823,1176823,Leptodactylus luctator,Wrestler Frog,Amphibia
3,1491113,1491113,Adenomera guarani,Guaraní leaf-litter frog,Amphibia
4,1595929,1595929,Lysapsus limellum,Uruguay Harlequin Frog,Amphibia


num classes: 234
train.csv rows: 35549


,primary_label,secondary_labels,type,latitude,longitude,scientific_name,common_name,class_name,inat_taxon_id,author,license,rating,url,filename,collection
0,1161364,[],[],-22.7562,-46.8666,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1216197....,1161364/iNat1216197.ogg,iNat
1,1161364,[],[],-22.7558,-46.8700,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/1114648....,1161364/iNat1114648.ogg,iNat
2,1161364,[],[],-22.7547,-46.8728,Guyalna cuta,Guyalna cuta,Insecta,1161364,Lucas Barbosa,cc-by-nc,0.0,https://static.inaturalist.org/sounds/810195.m...,1161364/iNat810195.ogg,iNat


species with train_audio samples: 206
median samples / species: 125


In [5]:
# Optional labeled train_soundscapes (5 s segments)
if TRAIN_SS_LABELS_CSV.exists():
    train_ss_labels = pd.read_csv(TRAIN_SS_LABELS_CSV)
    print('train_soundscapes labels:', len(train_ss_labels))
    display(train_ss_labels.head(3))
else:
    train_ss_labels = None
    print('No labeled soundscapes file found.')

train_soundscapes labels: 1478


,filename,start,end,primary_label
0,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:00,00:00:05,22961;23158;24321;517063;65380
1,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:05,00:00:10,22961;23158;24321;517063;65380
2,BC2026_Train_0039_S22_20211231_201500.ogg,00:00:10,00:00:15,22961;23158;24321;517063;65380


## 3. Audio → log-mel & on-disk cache
For each clip we precompute the log-mel of the loudest 5 s window once and save it to `mel_cache/`. Training then just memory-maps tiny `.npy` files — orders of magnitude faster than decoding `.ogg` every epoch.

In [6]:
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
    n_mels=N_MELS, f_min=FMIN, f_max=FMAX, power=2.0,
)
db_transform = torchaudio.transforms.AmplitudeToDB(stype='power', top_db=80)

def load_audio(path, sr=SR):
    """Load mono audio at target sr. Falls back to librosa on torchaudio failure."""
    try:
        wav, file_sr = torchaudio.load(str(path))
        if wav.shape[0] > 1:
            wav = wav.mean(dim=0, keepdim=True)
        if file_sr != sr:
            wav = torchaudio.functional.resample(wav, file_sr, sr)
        return wav.squeeze(0).numpy()
    except Exception:
        y, _ = librosa.load(str(path), sr=sr, mono=True)
        return y

def fix_length(y, n_samples=CHUNK_SAMPLES):
    """Pad or take the loudest n_samples window."""
    if len(y) >= n_samples:
        if len(y) > n_samples:
            hop = SR  # 1 s stride
            energies = [np.abs(y[i:i+n_samples]).mean()
                        for i in range(0, len(y) - n_samples + 1, hop)]
            start = int(np.argmax(energies)) * hop
            y = y[start:start + n_samples]
        else:
            y = y[:n_samples]
    else:
        y = np.pad(y, (0, n_samples - len(y)))
    return y.astype(np.float32)

def waveform_to_logmel(y):
    t = torch.from_numpy(y).float().unsqueeze(0)
    mel = mel_transform(t)
    logmel = db_transform(mel).squeeze(0)
    logmel = (logmel - logmel.mean()) / (logmel.std() + 1e-6)
    return logmel  # [N_MELS, T]

def cache_path_for(rel_filename: str) -> Path:
    """Stable cache filename derived from the source filename."""
    h = hashlib.md5(rel_filename.encode()).hexdigest()[:16]
    return CACHE_DIR / f'{h}.npy'

def compute_and_cache(rel_filename: str) -> Path:
    out = cache_path_for(rel_filename)
    if out.exists():
        return out
    try:
        y = load_audio(TRAIN_AUDIO_DIR / rel_filename)
        y = fix_length(y)
        spec = waveform_to_logmel(y).numpy().astype(np.float32)
    except Exception:
        spec = np.zeros((N_MELS, int(np.ceil(CHUNK_SAMPLES / HOP_LENGTH)) + 1),
                        dtype=np.float32)
    np.save(out, spec)
    return out

# Quick sanity check
if TRAIN_AUDIO_DIR.exists():
    rel = train_df.iloc[0]['filename']
    p = compute_and_cache(rel)
    spec = np.load(p)
    print('logmel shape:', spec.shape, '| dtype:', spec.dtype)

logmel shape: (128, 313) | dtype: float32


## 4. Dataset & stratified train / val / test split

We split clips per species so each species (with ≥3 samples) appears in all three sets. Species with only 1–2 samples go entirely to train.

In [7]:
# Subsample to keep CPU training tractable: at most MAX_FILES_PER_SP per species,
# preferring higher-rated clips.
subset = (train_df
          .sort_values('rating', ascending=False)
          .groupby('primary_label', group_keys=False)
          .head(MAX_FILES_PER_SP)
          .reset_index(drop=True))
subset = subset[subset['primary_label'].isin(label2idx)].reset_index(drop=True)
print('training subset size:', len(subset))

# Stratified 3-way split by primary_label.
rng = np.random.RandomState(SEED)
train_idx, val_idx, test_idx = [], [], []
for sp, grp in subset.groupby('primary_label'):
    idx = grp.index.to_numpy().copy()
    rng.shuffle(idx)
    n = len(idx)
    if n < 3:
        train_idx.extend(idx.tolist())
        continue
    n_test = max(1, int(round(n * TEST_FRAC)))
    n_val  = max(1, int(round(n * VAL_FRAC)))
    if n_test + n_val >= n:
        n_test, n_val = 1, 1
    test_idx.extend(idx[:n_test].tolist())
    val_idx.extend(idx[n_test:n_test + n_val].tolist())
    train_idx.extend(idx[n_test + n_val:].tolist())

train_subset = subset.loc[train_idx].reset_index(drop=True)
val_subset   = subset.loc[val_idx].reset_index(drop=True)
test_subset  = subset.loc[test_idx].reset_index(drop=True)
print(f'train: {len(train_subset)}  |  val: {len(val_subset)}  |  test: {len(test_subset)}')
print('val species coverage :', val_subset["primary_label"].nunique(),  '/', NUM_CLASSES)
print('test species coverage:', test_subset["primary_label"].nunique(), '/', NUM_CLASSES)

# ---- Precompute mel cache for everything we'll use (first run is slow, later runs are instant) ----
from concurrent.futures import ThreadPoolExecutor
all_files = pd.concat([train_subset, val_subset, test_subset])['filename'].unique().tolist()
to_make = [f for f in all_files if not cache_path_for(f).exists()]
print(f'spectrograms to precompute: {len(to_make)} / {len(all_files)}')
if to_make:
    with ThreadPoolExecutor(max_workers=max(2, os.cpu_count() // 2)) as ex:
        for i, _ in enumerate(ex.map(compute_and_cache, to_make), 1):
            if i % 100 == 0 or i == len(to_make):
                print(f'  cached {i}/{len(to_make)}')

# ---- Dataset reading from the cache ----
class CachedSpecDataset(Dataset):
    def __init__(self, df, label2idx, num_classes, train=False):
        self.df = df.reset_index(drop=True)
        self.label2idx = label2idx
        self.num_classes = num_classes
        self.train = train
        # SpecAugment parameters
        self.freq_mask = torchaudio.transforms.FrequencyMasking(freq_mask_param=24)
        self.time_mask = torchaudio.transforms.TimeMasking(time_mask_param=40)

    def __len__(self):
        return len(self.df)

    def _make_target(self, row):
        t = torch.zeros(self.num_classes, dtype=torch.float32)
        prim = row['primary_label']
        if prim in self.label2idx:
            t[self.label2idx[prim]] = 1.0
        sec = row.get('secondary_labels', '[]')
        if isinstance(sec, str) and sec.strip() not in ('', '[]', 'nan'):
            try:
                for s in json.loads(sec.replace("'", '"')):
                    if s in self.label2idx:
                        t[self.label2idx[s]] = 0.5     # soft label for secondary
            except Exception:
                pass
        return t

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        spec = np.load(cache_path_for(row['filename']))
        x = torch.from_numpy(spec).unsqueeze(0)              # [1, n_mels, T]
        if self.train and USE_SPEC_AUGMENT:
            x = self.freq_mask(x)
            x = self.time_mask(x)
        # 3-channel input for pretrained ImageNet backbone
        x = x.repeat(3, 1, 1)
        return x, self._make_target(row)

train_ds = CachedSpecDataset(train_subset, label2idx, NUM_CLASSES, train=True)
val_ds   = CachedSpecDataset(val_subset,   label2idx, NUM_CLASSES, train=False)
test_ds  = CachedSpecDataset(test_subset,  label2idx, NUM_CLASSES, train=False)

# ---- Class-balanced sampler: weight each sample by 1/sqrt(class freq) ----
train_labels = train_subset['primary_label'].map(label2idx).values
class_counts = np.bincount(train_labels, minlength=NUM_CLASSES)
class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
sample_weights = class_weights[train_labels]
sampler = WeightedRandomSampler(weights=sample_weights.tolist(),
                                num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, drop_last=True)
val_loader   = DataLoader(val_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

xb, yb = next(iter(train_loader))
print('batch:', xb.shape, yb.shape)

training subset size: 4494
train: 3054  |  val: 720  |  test: 720
val species coverage : 199 / 234
test species coverage: 199 / 234
spectrograms to precompute: 4493 / 4494
  cached 100/4493
  cached 200/4493
  cached 300/4493
  cached 400/4493
  cached 500/4493
  cached 600/4493
  cached 700/4493
  cached 800/4493
  cached 900/4493
  cached 1000/4493
  cached 1100/4493
  cached 1200/4493
  cached 1300/4493
  cached 1400/4493
  cached 1500/4493
  cached 1600/4493
  cached 1700/4493
  cached 1800/4493
  cached 1900/4493
  cached 2000/4493
  cached 2100/4493
  cached 2200/4493
  cached 2300/4493
  cached 2400/4493
  cached 2500/4493
  cached 2600/4493
  cached 2700/4493
  cached 2800/4493
  cached 2900/4493
  cached 3000/4493
  cached 3100/4493
  cached 3200/4493
  cached 3300/4493
  cached 3400/4493
  cached 3500/4493
  cached 3600/4493
  cached 3700/4493
  cached 3800/4493
  cached 3900/4493
  cached 4000/4493
  cached 4100/4493
  cached 4200/4493
  cached 4300/4493
  cached 4400/4493
 

## 5. Model — EfficientNet-B0 (pretrained, CPU-optimized)
~5.3M params with ImageNet-pretrained features. Faster CPU training than MobileNetV3-Small while maintaining better accuracy. Input is the 3-channel-repeated log-mel spectrogram. We replace the classifier with a 234-way head.

In [8]:
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

def build_model_efficientnet(num_classes):
    """
    Lightweight EfficientNet-B0: ~5.3M params, faster CPU inference than MobileNetV3-Small.
    Better accuracy with same speed on CPU for audio spectrograms.
    """
    try:
        weights = EfficientNet_B0_Weights.IMAGENET1K_V1
        net = efficientnet_b0(weights=weights)
        print('loaded ImageNet pretrained weights for EfficientNet-B0')
    except Exception as e:
        print('pretrained weights unavailable, training from scratch:', e)
        net = efficientnet_b0(weights=None)
    in_features = net.classifier[1].in_features
    net.classifier[1] = nn.Linear(in_features, num_classes)
    return net

model = build_model_efficientnet(NUM_CLASSES).to(DEVICE)
print('EfficientNet-B0 selected for faster CPU training')

print(sum(p.numel() for p in model.parameters()), 'parameters')


Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\wikto/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100.0%


loaded ImageNet pretrained weights for EfficientNet-B0
EfficientNet-B0 selected for faster CPU training
4307302 parameters


## 6. Train with validation metrics

Each epoch we report on the validation split:
- **comp_auc** — the official Kaggle metric (macro ROC-AUC over present species).
- **macro_map** (mean average precision) — robust to class imbalance.
- **top-1 accuracy** — does the highest-scoring class match any positive label?

Training enhancements over the baseline:
- mixup (α=0.4) — strong regularizer for audio.
- SpecAugment (freq + time masking) on the input.
- WeightedRandomSampler — over-samples rare species.
- AdamW + cosine LR schedule.
- Best-by-comp-AUC checkpoint is kept.

In [9]:
import pandas.api.types
import sklearn.metrics

class ParticipantVisibleError(Exception):
    pass

def competition_score(solution: pd.DataFrame, submission: pd.DataFrame,
                      row_id_column_name: str) -> float:
    """Verbatim Kaggle metric — macro ROC-AUC ignoring classes with no positives."""
    solution = solution.copy()
    submission = submission.copy()
    del solution[row_id_column_name]
    del submission[row_id_column_name]

    if not pandas.api.types.is_numeric_dtype(submission.values):
        bad = {x: submission[x].dtype for x in submission.columns
               if not pandas.api.types.is_numeric_dtype(submission[x])}
        raise ParticipantVisibleError(f'Invalid submission data types found: {bad}')

    solution_sums = solution.sum(axis=0)
    scored_columns = list(solution_sums[solution_sums > 0].index.values)
    assert len(scored_columns) > 0
    return sklearn.metrics.roc_auc_score(
        solution[scored_columns].values,
        submission[scored_columns].values,
        average='macro',
    )

def score_from_arrays(targets: np.ndarray, probs: np.ndarray) -> float:
    """Convenience wrapper: build the two DataFrames the metric expects."""
    # The metric only counts presence (>0); convert soft secondary labels to 0/1.
    sol = pd.DataFrame((targets > 0).astype(int), columns=SPECIES)
    sub = pd.DataFrame(probs, columns=SPECIES)
    sol.insert(0, 'row_id', np.arange(len(sol)))
    sub.insert(0, 'row_id', np.arange(len(sub)))
    return float(competition_score(sol, sub, 'row_id'))

## 6. Train with validation metrics

Metrics tracked each epoch on the validation set:
- **macro ROC-AUC** — the competition's headline metric (averages per-species AUC; only species present in val contribute).
- **macro mAP** (mean average precision) — robust to class imbalance.
- **top-1 accuracy** — does the highest-scoring class match `primary_label`?

In [10]:
from sklearn.metrics import average_precision_score

criterion = nn.BCEWithLogitsLoss()
optim = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim, T_max=EPOCHS * max(1, len(train_loader)))

def mixup_batch(x, y, alpha=MIXUP_ALPHA):
    if alpha <= 0:
        return x, y
    lam = np.random.beta(alpha, alpha)
    perm = torch.randperm(x.size(0))
    x = lam * x + (1 - lam) * x[perm]
    y = torch.clamp(lam * y + (1 - lam) * y[perm], 0.0, 1.0)
    return x, y

def train_one_epoch(loader):
    model.train()
    losses = []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        xb, yb = mixup_batch(xb, yb, MIXUP_ALPHA)
        logits = model(xb)
        loss = criterion(logits, yb)
        optim.zero_grad(); loss.backward(); optim.step()
        scheduler.step()
        losses.append(loss.item())
    return float(np.mean(losses)) if losses else float('nan')

@torch.no_grad()
def evaluate(loader):
    model.eval()
    all_logits, all_targets, losses = [], [], []
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = model(xb)
        losses.append(criterion(logits, yb).item())
        all_logits.append(logits.cpu().numpy())
        all_targets.append(yb.cpu().numpy())
    if not all_logits:
        return {'loss': float('nan'), 'comp_auc': float('nan'),
                'macro_map': float('nan'), 'top1_acc': float('nan'),
                'probs': None, 'targets': None}
    logits  = np.concatenate(all_logits, 0)
    targets = np.concatenate(all_targets, 0)
    probs   = 1 / (1 + np.exp(-logits))

    try:
        comp_auc = score_from_arrays(targets, probs)
    except Exception:
        comp_auc = float('nan')

    present = (targets > 0).any(0)
    if present.any():
        try:
            macro_map = average_precision_score((targets[:, present] > 0).astype(int),
                                                probs[:, present], average='macro')
        except ValueError:
            macro_map = float('nan')
    else:
        macro_map = float('nan')

    top1 = probs.argmax(1)
    top1_acc = float(np.mean([targets[i, top1[i]] > 0 for i in range(len(top1))]))
    return {'loss': float(np.mean(losses)), 'comp_auc': float(comp_auc),
            'macro_map': float(macro_map), 'top1_acc': top1_acc,
            'probs': probs, 'targets': targets}

history = []
best_score, best_state = -1.0, None
for ep in range(1, EPOCHS + 1):
    tl = train_one_epoch(train_loader)
    val = evaluate(val_loader)
    lr_now = optim.param_groups[0]['lr']
    history.append({'epoch': ep, 'lr': lr_now, 'train_loss': tl,
                    'val_loss': val['loss'], 'val_comp_auc': val['comp_auc'],
                    'val_map': val['macro_map'], 'val_top1': val['top1_acc']})
    print(f"epoch {ep}: lr={lr_now:.2e}  train_loss={tl:.4f}  "
          f"val_loss={val['loss']:.4f}  comp_auc={val['comp_auc']:.4f}  "
          f"map={val['macro_map']:.4f}  top1={val['top1_acc']:.4f}")
    if val['comp_auc'] == val['comp_auc'] and val['comp_auc'] > best_score:
        best_score = val['comp_auc']
        best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
torch.save(model.state_dict(), 'mobilenetv3_small_birdclef.pt')
print(f'saved checkpoint | best val comp_auc = {best_score:.4f}')

pd.DataFrame(history)

epoch 1: lr=2.80e-04  train_loss=0.1151  val_loss=0.0284  comp_auc=0.5037  map=0.0142  top1=0.0069
epoch 2: lr=2.25e-04  train_loss=0.0284  val_loss=0.0282  comp_auc=0.5065  map=0.0142  top1=0.0069
epoch 3: lr=1.50e-04  train_loss=0.0282  val_loss=0.0280  comp_auc=0.5128  map=0.0156  top1=0.0083
epoch 4: lr=7.50e-05  train_loss=0.0281  val_loss=0.0279  comp_auc=0.5233  map=0.0154  top1=0.0056
epoch 5: lr=2.01e-05  train_loss=0.0281  val_loss=0.0279  comp_auc=0.5236  map=0.0164  top1=0.0125
epoch 6: lr=0.00e+00  train_loss=0.0280  val_loss=0.0279  comp_auc=0.5218  map=0.0171  top1=0.0139
saved checkpoint | best val comp_auc = 0.5236


,epoch,lr,train_loss,val_loss,val_comp_auc,val_map,val_top1
0,1,0.000280,0.115086,0.028427,0.503748,0.014176,0.006944
1,2,0.000225,0.028388,0.028162,0.506494,0.014164,0.006944
2,3,0.000150,0.028217,0.028008,0.512794,0.015597,0.008333
3,4,0.000075,0.028149,0.027941,0.523268,0.015424,0.005556
4,5,0.000020,0.028074,0.027919,0.523587,0.016362,0.012500
5,6,0.000000,0.027979,0.027928,0.521773,0.017081,0.013889


## 6b. Held-out test-set evaluation

The test split was never seen during training or model selection. These numbers are an honest estimate of generalization on the *same distribution as train_audio* — the actual competition test set (passive recordings) will likely score lower due to domain shift.

In [11]:
def predict_soundscape(path, model):
    y = load_audio(path)
    total = len(y)
    rows = []
    fname = Path(path).name
    model.eval()
    with torch.no_grad():
        for end_sec in range(CHUNK_SECONDS, 61, CHUNK_SECONDS):
            start = (end_sec - CHUNK_SECONDS) * SR
            end   = end_sec * SR
            seg = y[start:end] if end <= total else np.pad(y[start:total], (0, end - total))
            if len(seg) < CHUNK_SAMPLES:
                seg = np.pad(seg, (0, CHUNK_SAMPLES - len(seg)))
            spec = waveform_to_logmel(seg.astype(np.float32))
            x = spec.unsqueeze(0).unsqueeze(0).repeat(1, 3, 1, 1)   # [1, 3, n_mels, T]
            probs = torch.sigmoid(model(x)).squeeze(0).cpu().numpy()
            rows.append((f'{fname[:-4]}_{end_sec}', probs))
    return rows

if TEST_SOUNDSCAPES.exists() and any(TEST_SOUNDSCAPES.iterdir()):
    out_rows = []
    for p in sorted(TEST_SOUNDSCAPES.glob('*.ogg')):
        for row_id, probs in predict_soundscape(p, model):
            out_rows.append([row_id, *probs.tolist()])
    sub = pd.DataFrame(out_rows, columns=['row_id', *SPECIES])
    sub.to_csv('submission.csv', index=False)
    print('wrote submission.csv', sub.shape)
else:
    print('test_soundscapes/ empty — only populated at scoring time on Kaggle.')

wrote submission.csv (0, 235)


## 7. Inference on test soundscapes (skeleton)
Each 1 min soundscape → 12 non-overlapping 5 s segments. Predict probabilities per segment.

In [12]:
def predict_soundscape(path, model):
    y = load_audio(path)
    total = len(y)
    rows = []
    fname = Path(path).name
    model.eval()
    with torch.no_grad():
        for end_sec in range(CHUNK_SECONDS, 61, CHUNK_SECONDS):
            start = (end_sec - CHUNK_SECONDS) * SR
            end   = end_sec * SR
            seg = y[start:end] if end <= total else np.pad(y[start:total], (0, end - total))
            seg = fix_length(seg)
            spec = waveform_to_logmel(seg).unsqueeze(0).unsqueeze(0)
            probs = torch.sigmoid(model(spec)).squeeze(0).cpu().numpy()
            rows.append((f'{fname[:-4]}_{end_sec}', probs))
    return rows

if TEST_SOUNDSCAPES.exists() and any(TEST_SOUNDSCAPES.iterdir()):
    out_rows = []
    for p in sorted(TEST_SOUNDSCAPES.glob('*.ogg')):
        for row_id, probs in predict_soundscape(p, model):
            out_rows.append([row_id, *probs.tolist()])
    sub = pd.DataFrame(out_rows, columns=['row_id', *SPECIES])
    sub.to_csv('submission.csv', index=False)
    print('wrote submission.csv', sub.shape)
else:
    print('test_soundscapes/ empty — only populated at scoring time on Kaggle.')

wrote submission.csv (0, 235)
